In [1]:
!pip install -q tiktoken

In [2]:
import os, requests
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

print("torch:", torch.__version__)
print("tiktoken:", tiktoken.__version__)

torch: 2.10.0+cpu
tiktoken: 0.12.0


In [3]:
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
if not os.path.exists("the-verdict.txt"):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    open("the-verdict.txt", "wb").write(r.content)

raw_text = open("the-verdict.txt", encoding="utf-8").read()
print("chars:", len(raw_text))
print(raw_text[:99])

chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(ids)
print(tokenizer.decode(ids))

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [5]:
enc_text = tokenizer.encode(raw_text)
print("story tokens:", len(enc_text))

enc_sample = enc_text[50:]
context_size = 4
for i in range(1, context_size + 1):
    ctx = enc_sample[:i]
    nxt = enc_sample[i]
    print(tokenizer.decode(ctx), "---->", tokenizer.decode([nxt]))

story tokens: 5145
 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    ds = GPTDatasetV1(txt, tiktoken.get_encoding("gpt2"), max_length, stride)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)

loader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(loader))
print("Inputs:\n", inputs)
print("Targets:\n", targets)

In [ ]:
vocab_size, output_dim, max_length = 50257, 256, 4  # GPT-2 vocab

token_emb = torch.nn.Embedding(vocab_size, output_dim)
pos_emb = torch.nn.Embedding(max_length, output_dim)

token_embeddings = token_emb(inputs)                    # [8, 4, 256]
pos_embeddings = pos_emb(torch.arange(max_length))      # [4, 256]
input_embeddings = token_embeddings + pos_embeddings    # [8, 4, 256]
print(input_embeddings.shape)